In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT_DIR = Path(r"C:\Users\mathlouthi\Desktop\POSS-LOGIC-LM-GITHUB")

LOGICLM_4VARIANTS_DIR = PROJECT_DIR / "LOGICLM-4Variants"
POSS_RESULTS_DIR = PROJECT_DIR / "POSS-LOGICLM"

OUTPUT_DIR = PROJECT_DIR / "analysis_outputs" / "final_figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIGURE_PNG = OUTPUT_DIR / "figure4_variants_comparison.png"
FIGURE_PDF = OUTPUT_DIR / "figure4_variants_comparison.pdf"
SOURCE_CSV = OUTPUT_DIR / "figure4_source_values.csv"
DIAGNOSTIC_CSV = OUTPUT_DIR / "figure4_diagnostic_files.csv"


# ============================================================
# 2. ORDRES D'AFFICHAGE
# ============================================================

DATASET_ORDER = ["PrOntoQA", "ProofWriter", "FOLIO"]
MODEL_ORDER = ["Gemma", "LLaMA", "Qwen"]

VARIANT_ORDER = [
    "LM-greedy",
    "LM-greedy+SR",
    "LMBeam",
    "LMBeam+SR",
    "POSS-LM",
]

MODEL_COLORS = {
    "Gemma": "#5B9BD5",
    "LLaMA": "#3CB371",
    "Qwen": "#E07B52",
}


# ============================================================
# 3. RACINES DES VARIANTES
# ============================================================

VARIANT_ROOTS = {
    "LM-greedy": LOGICLM_4VARIANTS_DIR / "LOGICLMGREEDY-NO-SR",
    "LM-greedy+SR": LOGICLM_4VARIANTS_DIR / "LOGICLMGREEDY+SR",
    "LMBeam": LOGICLM_4VARIANTS_DIR / "LogicLM-NOSR-F1BEAM",
    "LMBeam+SR": LOGICLM_4VARIANTS_DIR / "LogicLM+SR-F1BEAM",
    "POSS-LM": POSS_RESULTS_DIR,
}


# ============================================================
# 4. ALIASES POUR RECONNAÎTRE LES FICHIERS
# ============================================================

DATASET_ALIASES = {
    "PrOntoQA": ["prontoqa", "pqa"],
    "ProofWriter": ["proofwriter", "pw"],
    "FOLIO": ["folio"],
}

MODEL_ALIASES = {
    "Gemma": ["gemma"],
    "LLaMA": ["llama", "llama3", "llama31", "llama_3"],
    "Qwen": ["qwen"],
}


# ============================================================
# 5. HELPERS TEXTE
# ============================================================

def normalize_text(value: Any) -> str:
    text = str(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return f" {text.strip()} "


def contains_alias(text: str, aliases: List[str]) -> bool:
    text_norm = normalize_text(text)

    for alias in aliases:
        alias_norm = normalize_text(alias).strip()
        if f" {alias_norm} " in text_norm:
            return True

        if alias_norm in text_norm:
            return True

    return False


def path_text(path: Path) -> str:
    return str(path).replace("\\", "/")


# ============================================================
# 6. LECTURE JSON ET ACCURACY
# ============================================================

def read_json(path: Path) -> Dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")

    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def get_results(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    for key in ["results", "data", "items", "examples", "records"]:
        value = data.get(key)
        if isinstance(value, list):
            return [row for row in value if isinstance(row, dict)]

    return []


def to_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value

    if isinstance(value, int):
        return value == 1

    if isinstance(value, float):
        return value == 1.0

    if isinstance(value, str):
        value_norm = value.strip().lower()
        return value_norm in {"true", "1", "yes", "correct", "ok"}

    return False


def compute_accuracy_from_results(data: Dict[str, Any]) -> float:
    results = get_results(data)

    if not results:
        raise ValueError("Aucune liste results/data/items/examples/records trouvée dans le JSON.")

    total = len(results)

    correct_keys = [
        "correct",
        "is_correct",
        "success",
        "is_success",
        "prediction_correct",
    ]

    correct = 0

    for row in results:
        row_correct = False

        for key in correct_keys:
            if key in row:
                row_correct = to_bool(row.get(key))
                break

        if row_correct:
            correct += 1

    return round(100.0 * correct / total, 2)


def get_accuracy(path: Path) -> float:
    data = read_json(path)
    stats = data.get("stats", {})

    if isinstance(stats, dict):
        for key in ["accuracy", "acc", "Accuracy"]:
            if key in stats:
                try:
                    return round(float(stats[key]), 2)
                except Exception:
                    pass

    return compute_accuracy_from_results(data)


# ============================================================
# 7. RECHERCHE AUTOMATIQUE DES FICHIERS
# ============================================================

def list_json_files(root: Path) -> List[Path]:
    if not root.exists():
        return []

    return sorted(root.rglob("*.json"))


def score_candidate(path: Path, dataset: str, model: str, variant: str) -> int:
    full = path_text(path)
    name = path.name

    dataset_aliases = DATASET_ALIASES[dataset]
    model_aliases = MODEL_ALIASES[model]

    score = 0

    if contains_alias(name, dataset_aliases):
        score += 6
    elif contains_alias(full, dataset_aliases):
        score += 3

    if contains_alias(name, model_aliases):
        score += 6
    elif contains_alias(full, model_aliases):
        score += 4

    full_norm = normalize_text(full)
    name_norm = normalize_text(name)

    if " result " in name_norm or " results " in name_norm:
        score += 2

    if variant == "POSS-LM":
        if " pbs " in name_norm or " pbs " in full_norm:
            score += 4
        if " logiclm " in name_norm:
            score -= 3

    if variant == "LM-greedy":
        if " nosr " in name_norm or " no sr " in full_norm:
            score += 2
        if " greedy " in name_norm or " greedy " in full_norm:
            score += 2

    if variant == "LM-greedy+SR":
        if " greedy " in name_norm or " greedy " in full_norm:
            score += 2
        if " nosr " in name_norm or " no sr " in full_norm:
            score -= 4

    if variant == "LMBeam":
        if " nosr " in name_norm or " no sr " in full_norm:
            score += 2
        if " beam " in full_norm or " f1beam " in full_norm:
            score += 2

    if variant == "LMBeam+SR":
        if " beam " in full_norm or " f1beam " in full_norm:
            score += 2
        if " nosr " in name_norm or " no sr " in full_norm:
            score -= 5

    return score


def find_result_file(root: Path, dataset: str, model: str, variant: str) -> Tuple[Optional[Path], List[Tuple[int, Path]]]:
    all_files = list_json_files(root)

    scored: List[Tuple[int, Path]] = []

    for path in all_files:
        score = score_candidate(path, dataset, model, variant)

        if score > 0:
            scored.append((score, path))

    scored.sort(key=lambda item: item[0], reverse=True)

    strong_candidates = [(score, path) for score, path in scored if score >= 8]

    if strong_candidates:
        return strong_candidates[0][1], scored[:10]

    return None, scored[:10]


def build_diagnostic_report() -> pd.DataFrame:
    rows = []

    for variant in VARIANT_ORDER:
        root = VARIANT_ROOTS[variant]

        for dataset in DATASET_ORDER:
            for model in MODEL_ORDER:
                selected, candidates = find_result_file(root, dataset, model, variant)

                rows.append({
                    "Variant": variant,
                    "Dataset": dataset,
                    "Model": model,
                    "Root": str(root),
                    "Selected": str(selected) if selected else "",
                    "Status": "FOUND" if selected else "MISSING",
                    "Top candidates": " | ".join(
                        [f"{score}: {path}" for score, path in candidates[:5]]
                    ),
                })

    return pd.DataFrame(rows)


# ============================================================
# 8. COLLECTE DES VALEURS
# ============================================================

def collect_values() -> pd.DataFrame:
    rows = []
    missing_rows = []

    for variant in VARIANT_ORDER:
        root = VARIANT_ROOTS[variant]

        if not root.exists():
            raise FileNotFoundError(
                f"Dossier introuvable pour {variant} : {root}\n"
                f"Vérifie le nom du dossier dans ton projet."
            )

        for dataset in DATASET_ORDER:
            for model in MODEL_ORDER:
                path, candidates = find_result_file(root, dataset, model, variant)

                if path is None:
                    missing_rows.append({
                        "Variant": variant,
                        "Dataset": dataset,
                        "Model": model,
                        "Root": str(root),
                        "Top candidates": " | ".join(
                            [f"{score}: {candidate}" for score, candidate in candidates[:5]]
                        ),
                    })
                    continue

                acc = get_accuracy(path)

                rows.append({
                    "Dataset": dataset,
                    "Model": model,
                    "Variant": variant,
                    "Accuracy": acc,
                    "Path": str(path),
                })

    if missing_rows:
        missing_df = pd.DataFrame(missing_rows)

        print("\n" + "=" * 100)
        print("FICHIERS MANQUANTS")
        print("=" * 100)
        print(missing_df.to_string(index=False))

        raise FileNotFoundError(
            "\nCertains fichiers JSON n'ont pas été retrouvés automatiquement.\n"
            f"Un diagnostic complet a été sauvegardé ici : {DIAGNOSTIC_CSV}\n"
            "Ouvre ce CSV pour voir les candidats trouvés et corrige le nom du dossier ou du fichier."
        )

    return pd.DataFrame(rows)


# ============================================================
# 9. FIGURE 4
# ============================================================

def make_figure4(df: pd.DataFrame) -> None:
    x = np.arange(len(VARIANT_ORDER))
    width = 0.24

    fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.8), sharey=True)

    for ax, dataset in zip(axes, DATASET_ORDER):
        for i, model in enumerate(MODEL_ORDER):
            values = []

            for variant in VARIANT_ORDER:
                row = df[
                    (df["Dataset"] == dataset)
                    & (df["Model"] == model)
                    & (df["Variant"] == variant)
                ]

                if row.empty:
                    raise ValueError(f"Valeur manquante : {dataset} / {model} / {variant}")

                values.append(float(row.iloc[0]["Accuracy"]))

            offset = (i - 1) * width

            bars = ax.bar(
                x + offset,
                values,
                width=width,
                color=MODEL_COLORS[model],
                edgecolor="white",
                linewidth=0.7,
                label=model,
            )

            bars[-1].set_hatch("///")
            bars[-1].set_edgecolor("white")

            for bar, value in zip(bars, values):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.8,
                    f"{value:.0f}",
                    ha="center",
                    va="bottom",
                    fontsize=7,
                )

        ax.set_title(dataset, fontsize=12, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(VARIANT_ORDER, rotation=0, ha="center", fontsize=8)
        ax.grid(axis="y", linestyle="--", alpha=0.3)
        ax.set_ylim(20, 105)

    axes[0].set_ylabel("Accuracy (%)", fontsize=10)

    handles, labels = axes[0].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, -0.02),
    )

    plt.tight_layout(rect=[0.02, 0.10, 1, 1])

    fig.savefig(FIGURE_PNG, dpi=300, bbox_inches="tight")
    fig.savefig(FIGURE_PDF, bbox_inches="tight")

    plt.show()

    print("\nFigure PNG :", FIGURE_PNG)
    print("Figure PDF :", FIGURE_PDF)


# ============================================================
# 10. MAIN
# ============================================================

def main() -> None:
    diagnostic_df = build_diagnostic_report()
    diagnostic_df.to_csv(DIAGNOSTIC_CSV, index=False, encoding="utf-8-sig")

    print("=" * 100)
    print("DIAGNOSTIC DES FICHIERS")
    print("=" * 100)
    print(diagnostic_df[["Variant", "Dataset", "Model", "Status", "Selected"]].to_string(index=False))

    df = collect_values()

    df.to_csv(SOURCE_CSV, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 100)
    print("VALEURS UTILISÉES POUR FIGURE 4")
    print("=" * 100)

    pivot = df.pivot_table(
        index=["Dataset", "Model"],
        columns="Variant",
        values="Accuracy",
        aggfunc="first",
    )[VARIANT_ORDER]

    print(pivot.to_string())

    print("\n" + "=" * 100)
    print("FICHIERS SOURCES UTILISÉS")
    print("=" * 100)
    print(df[["Dataset", "Model", "Variant", "Accuracy", "Path"]].to_string(index=False))

    make_figure4(df)


if __name__ == "__main__":
    main()